In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from rdkit import Chem
from rdkit import RDLogger
RDLogger.logger().setLevel(RDLogger.CRITICAL)

C:\Users\noton\miniconda3\envs\OPSdesign2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def canon(s: str):
    if not s or not s.strip():
        return None
    try:
        m = Chem.MolFromSmiles(s)
        if m is None:
            return None
        return Chem.MolToSmiles(m, canonical=True, isomericSmiles=True)
    except Exception:
        return None

In [3]:
torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("Frag2OPST5_1M/tokenizer")
model = AutoModelForSeq2SeqLM.from_pretrained("Frag2OPST5_1M/model").to(device)
model.eval()
model.config.use_cache = True

if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
test_enc = torch.load("../tokenizer/pretraining/test_enc.pt", weights_only=False)

labels_all = test_enc["labels"].clone()
labels_all[labels_all == -100] = tokenizer.pad_token_id
gold_all = tokenizer.batch_decode(labels_all, skip_special_tokens=True)


bs = 32
topk = 5
pred_all_topk = []
for i in range(0, len(gold_all), bs):
    input_ids      = test_enc["input_ids"][i:i+bs].to(device)
    attention_mask = test_enc["attention_mask"][i:i+bs].to(device)

    with torch.no_grad():
        gen_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=128,
            num_beams=topk,
            num_return_sequences=topk,
            do_sample=False,
            early_stopping=True,
        )

    decoded = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)
    for j in range(0, len(decoded), topk):
        pred_all_topk.append(decoded[j:j+topk])

def contains_exact_hit(preds, gold, K):
    g = gold.strip()
    return g in {p.strip() for p in preds[:K]}

def contains_valid(preds, K):
    return any(canon(p) is not None for p in preds[:K])

N = len(gold_all)

# Top-1
exact_top1 = sum(p[0].strip() == g.strip() for p, g in zip(pred_all_topk, gold_all))
valid1     = sum(contains_valid(p, 1) for p in pred_all_topk)
print(f"Top-1 Exact: {exact_top1/N:.3%} ({exact_top1}/{N})")
print(f"Top-1 Valid: {valid1/N:.3%} ({valid1}/{N})\n")

# Top-3
exact_top3 = sum(contains_exact_hit(p, g, 3) for p, g in zip(pred_all_topk, gold_all))
valid3     = sum(contains_valid(p, 3) for p in pred_all_topk)
print(f"Top-3 Exact: {exact_top3/N:.3%} ({exact_top3}/{N})")
print(f"Top-3 Valid: {valid3/N:.3%} ({valid3}/{N})\n")

# Top-5
exact_top5 = sum(contains_exact_hit(p, g, 5) for p, g in zip(pred_all_topk, gold_all))
valid5     = sum(contains_valid(p, 5) for p in pred_all_topk)
print(f"Top-5 Exact: {exact_top5/N:.3%} ({exact_top5}/{N})")
print(f"Top-5 Valid: {valid5/N:.3%} ({valid5}/{N})")

Top-1 Exact: 42.714% (21357/50000)
Top-1 Valid: 98.168% (49084/50000)

Top-3 Exact: 66.534% (33267/50000)
Top-3 Valid: 98.522% (49261/50000)

Top-5 Exact: 71.690% (35845/50000)
Top-5 Valid: 98.654% (49327/50000)
